# 00. Python Primer for Engineers
**Engineering Data Processing with Python** | one-day course | approx. 50 minutes

> **EXERCISE NOTEBOOK.** Work top to bottom. Run every cell in order (Shift+Enter). Where you see `____` replace it with your answer. Where you see `# TODO`, write the code. If an error mentions `____` (for example `NameError: name '____' is not defined`), you have a blank left to fill. Blanks are four underscores.


In [ ]:
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = ""   # if your trainer gives you a download link for data.zip, paste it between the quotes

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

## What this notebook is for

Some of you have written Python before, some of you have not. This primer gets everyone to the same starting line
before we touch real engineering data. If you already know Python, move quickly: run the cells, do the exercises,
and help your neighbour.

Everything here uses small engineering examples (motors, pumps, sensors) so you can check answers with your own
engineering judgement. That habit matters more than any syntax.

**How the exercises work today**

| Label | What you do |
|---|---|
| **Level 1: fill the blanks** | Most of the code is written. Replace each `____` with the missing piece. |
| **Level 2: finish the code** | The structure is there. Write the lines marked `# TODO`. |
| **Level 3: write it yourself** | You write it, and you can use Gemini to help. Then you check it. |

After most exercises there is a **check cell**. Run it. It prints `PASS` or `NOT YET` with a hint.

## 1. Colab in two minutes

* A notebook is a list of **cells**. Grey cells are code, white cells are text.
* Run a cell: click it and press **Shift + Enter**. The number in `[ ]` on the left shows the order cells ran in.
* **Order matters.** If a cell uses a variable, the cell that created it must have run first.
* Broke something? **Runtime > Restart session**, then run from the top. You cannot damage anything.
* The **Files** panel (folder icon on the left) shows the files in this session.
* Each Colab notebook runs on its own temporary machine. Files you create disappear when the session ends,
  and a different notebook cannot see them. Download anything you want to keep.

In [ ]:
print("Hello from Colab")
2 + 2   # the last line of a cell is displayed automatically

## 2. Variables and arithmetic

A variable is a name for a value. Put units in the name: `current_a` beats `c`. Future you will thank you.

Three-phase motor input power: $P = \sqrt{3} \, V I \cos\phi$

In [ ]:
voltage_v = 400
current_a = 32
power_factor = 0.85

power_kw = (3 ** 0.5) * voltage_v * current_a * power_factor / 1000   # ** means "to the power of"
print(power_kw)
print(f"Motor input power: {power_kw:.1f} kW")   # an f-string: values go inside { }, :.1f means 1 decimal place

### Exercise 0.1 [Level 1: fill the blanks]

Hydraulic power of a pump: $P = \rho g Q H$ (in watts). Work it out in kW for water at 0.05 m³/s against a 30 m head.
Replace each `____`.

In [ ]:
density_kg_m3 = 1000
g = 9.81
flow_m3_s = ____
head_m = ____

hydraulic_power_kw = ____ / 1000
print(f"Hydraulic power: {hydraulic_power_kw:.2f} kW")

In [ ]:
check(abs(hydraulic_power_kw - 14.715) < 0.01, "14.72 kW. Sanity check: a small process pump, sounds right.",
      "Expected about 14.7 kW. Check flow, head and the formula.")

## 3. Types: numbers, text and the "15 kW" problem

Every value has a type: `int` (whole number), `float` (decimal), `str` (text), `bool` (True/False).
Most messy-data pain comes from numbers that arrived as **text**, for example `"15 kW"` exported from a spreadsheet.

In [ ]:
rated_power = "15 kW"
print(type(rated_power))

try:
    rated_power + 5          # text + number does not work
except TypeError as err:
    print("TypeError:", err)

# Fix: remove the unit, tidy the spaces, convert to a number
rated_power_kw = float(rated_power.replace("kW", "").strip())
print(rated_power_kw + 5)

When you hit a real error you will see a red block of text (a *traceback*). **Read the last line first.** It tells you
the type of error and usually the reason. The lines above it tell you where it happened.

### Exercise 0.2 [Level 1: fill the blanks]

Asset IDs typed by hand often have stray spaces and the wrong case. Turn `"  cnc-009 "` into `"CNC-009"`.
Text has handy *methods*: `.strip()` removes spaces at both ends, `.upper()` makes capitals, `.lower()` makes lower case.

In [ ]:
raw_id = "  cnc-009 "
clean_id = raw_id.____().____()
print(repr(clean_id))   # repr() shows the quotes, so you can see any hidden spaces

In [ ]:
check(clean_id == "CNC-009", "Clean ID.", "Expected 'CNC-009' with no spaces.")

## 4. Lists

A list holds several values in order. Square brackets. Position counting starts at **0**.

In [ ]:
bearing_temp_c = [61.2, 63.5, 62.8, 70.4, 66.1, 64.0]

print("Number of readings:", len(bearing_temp_c))
print("First reading:", bearing_temp_c[0])
print("Last reading:", bearing_temp_c[-1])        # negative counts from the end
print("First three:", bearing_temp_c[0:3])        # a slice: from position 0 up to (not including) 3
print("Max:", max(bearing_temp_c), " Min:", min(bearing_temp_c))

bearing_temp_c.append(65.3)                        # add a new reading on the end
print(bearing_temp_c)

### Exercise 0.3 [Level 1: fill the blanks]

Vibration readings from a pump in mm/s. Work out the average and the maximum.
Hint: average = total divided by count. `sum()` gives the total.

In [ ]:
vibration_mm_s = [2.1, 2.4, 2.2, 7.8, 2.3, 9.4, 2.5, 8.1, 2.2]

average_vibration = ____ / ____
max_vibration = ____
print(f"Average {average_vibration:.2f} mm/s, max {max_vibration} mm/s")

In [ ]:
check(round(average_vibration, 2) == 4.33 and max_vibration == 9.4, "Average 4.33, max 9.4.",
      "Expected average 4.33 and max 9.4.")

Notice the average (4.33) hides the fact that three readings are way above the others.
Summary numbers are useful, but always look at the raw values too.

## 5. Loops and decisions

`for` repeats something for each item. `if` makes a decision. **Indentation (4 spaces) is part of the code**:
it tells Python which lines belong inside the loop or the `if`.

In [ ]:
TEMP_LIMIT_C = 65

for t in bearing_temp_c:
    if t > TEMP_LIMIT_C:
        print(f"{t} C  -> ABOVE LIMIT")
    else:
        print(f"{t} C  ok")

### Exercise 0.4 [Level 2: finish the code]

Build a list called `alarms` containing every vibration reading above the alarm limit.
The loop is there. Write the `# TODO` part. You need `if` and `.append()`.

*(7.1 mm/s is an illustrative limit for today. In real life it comes from your machine class and standard, e.g. ISO 20816.)*

In [ ]:
ALARM_LIMIT_MM_S = 7.1
alarms = []

for v in vibration_mm_s:
    # TODO: if v is above ALARM_LIMIT_MM_S, append v to the alarms list
    pass   # delete this line once you have written your code

print("Alarm readings:", alarms)
print("Number of alarms:", len(alarms))

In [ ]:
check(alarms == [7.8, 9.4, 8.1], "Three alarms found.", "Expected [7.8, 9.4, 8.1].")

## 6. Dictionaries

A dictionary stores values against **keys** (names). Curly brackets. Think of one row of an asset register.

In [ ]:
asset = {
    "asset_id": "CMP-001",
    "asset_type": "Air Compressor",
    "site": "Cork",
    "rated_power_kw": 55.0,
}
print(asset["site"])
print(asset.get("criticality", "not recorded"))   # .get() gives a default instead of an error if the key is missing

for key, value in asset.items():
    print(f"{key:15} {value}")

### Exercise 0.5 [Level 1: fill the blanks]

Add a `criticality` of `"A"` to the asset, then read back its rated power.

In [ ]:
asset[____] = "A"
power = asset[____]
print(asset)
print(power)

In [ ]:
check(asset.get("criticality") == "A" and power == 55.0, "Dictionary updated.", "Check the key names (they are case sensitive).")

## 7. Functions

A function packages a calculation so you can reuse it and test it. `def` defines it, `return` sends the answer back.
Functions are how we will turn today's notebooks into a repeatable script later.

In [ ]:
def celsius_to_fahrenheit(temp_c):
    """Convert a temperature from degrees C to degrees F."""
    return temp_c * 9 / 5 + 32

print(celsius_to_fahrenheit(100))
print([celsius_to_fahrenheit(t) for t in [0, 37, 100]])   # a "list comprehension": a loop that builds a list in one line

### Exercise 0.6 [Level 1: fill the blanks]

Simulation tools often export temperatures in kelvin. Complete `kelvin_to_celsius`.

In [ ]:
def kelvin_to_celsius(temp_k):
    """Convert kelvin to degrees C."""
    return ____

print(kelvin_to_celsius(353.15))

In [ ]:
check(abs(kelvin_to_celsius(353.15) - 80) < 1e-9, "353.15 K is 80 C.", "Subtract 273.15.")

### Exercise 0.7 [Level 2: finish the code]

Write two small functions. Keep them to one line each inside.

In [ ]:
def watts_to_kw(power_w):
    # TODO: return the power in kW
    return power_w   # placeholder, replace this line

def fahrenheit_to_celsius(temp_f):
    # TODO: return the temperature in C.  C = (F - 32) * 5 / 9
    return temp_f    # placeholder, replace this line

print(watts_to_kw(15000), fahrenheit_to_celsius(212))

In [ ]:
check(watts_to_kw(15000) == 15 and abs(fahrenheit_to_celsius(212) - 100) < 1e-9,
      "Both conversions work.", "15000 W should give 15 kW and 212 F should give 100 C.")

## 8. A real file: a boiler data dictionary (JSON)

`data/boiler.json` describes the variables in a boiler simulation export. JSON is just dictionaries and lists
written as text, so everything above applies. This sort of file is common: a *data dictionary* that tells you
names and units.

In [ ]:
import json

with open("data/boiler.json") as f:
    boiler = json.load(f)

print(type(boiler))
print(boiler.keys())
print("Sample frequency:", boiler["sample_frequency"])
print("First variable:", boiler["variables"][1])

In [ ]:
# Loop over the variables and print name, unit used in the export, and the unit in the source model
for var in boiler["variables"]:
    print(f"{var['Name']:15} export unit: {var['Unit']:6} model unit: {var['Modelica.Unit']}")

### Exercise 0.8 [Level 2: finish the code]

Some variables were converted between the model and the export (W to kW, K to °C). If you ever join this data to
raw model output, you need to know which. Build a list `needs_conversion` of `(name, model_unit, export_unit)`
for every variable where the two units differ.

In [ ]:
needs_conversion = []

for var in boiler["variables"]:
    # TODO: if var["Unit"] is different from var["Modelica.Unit"] (use != for "not equal"),
    #       append a tuple (var["Name"], var["Modelica.Unit"], var["Unit"]) to needs_conversion
    pass

for item in needs_conversion:
    print(item)

In [ ]:
check(len(needs_conversion) == 3 and needs_conversion[0] == ("P_fuel_kW", "W", "kW"),
      "Three variables changed units: two powers (W to kW) and the outlet temperature (K to degC).",
      "Expected 3 items, the first being ('P_fuel_kW', 'W', 'kW').")

## 9. Working with Gemini in Colab (AI pair programming)

Colab has Gemini built in. Used well it is a patient tutor and a fast typist. Used badly it writes plausible code
you do not understand, which then produces plausible numbers that are wrong. In engineering that second outcome is
the dangerous one. A crash is easy to spot. A wrong but tidy answer is not.

**Where to find it** (Google moves buttons around, so look for the Gemini icon):

* **Gemini chat panel**: ask questions about your notebook, your data or an error.
* **Learn Mode** (toggle in the Gemini chat): Gemini acts as a tutor and guides you step by step instead of handing
  you the full answer. **Use this for Level 1 and 2 exercises today.**
* **Generate code** in an empty cell: describe what you want in plain English.
* **Explain / fix error**: appears under a cell that failed. It proposes a change you can review before accepting.
* **Custom Instructions**: saved in the notebook and applied to every Gemini chat in it.

**Suggested Custom Instructions for today** (paste into Gemini's custom instructions):

```
I am an engineer learning Python and pandas for cleaning engineering data.
Explain in plain English and keep code short with comments.
Use pandas. Do not invent column names; if you need them, ask me to run df.columns or df.head().
Before giving code, state what you are assuming about my data.
After giving code, tell me one way to check the result is correct.
```

**A prompt pattern that works: Context, Task, Constraints, Check**

* *Context*: what the data is. Paste `df.head()` and `df.dtypes`, not the whole dataset.
* *Task*: one small step at a time ("convert the cost column to numbers"), not "clean my data".
* *Constraints*: "keep the original column", "do not drop rows", "use pandas only".
* *Check*: "how do I verify this worked?" and **"what did you assume?"**

**Rules for using AI on engineering data**

1. Read the code before you run it. If you cannot explain a line, ask Gemini to explain it.
2. Check the result against something you know: row counts before and after, a few rows by eye, a hand calculation.
3. Ask "what could you not determine, and what did you assume?" Assumptions are where wrong answers hide.
4. Company data: follow your organisation's AI policy. Do not paste confidential or personal data into AI tools
   unless your organisation has approved that tool for that data. Today's datasets are fictional or public.
5. "Vibe coding" (describe it, accept what comes back) is fine for a throwaway chart. Anything that feeds a
   decision, a report or a maintenance plan needs you to understand every line.

### Exercise 0.9 [Level 3: use Gemini's error help]

The cell below has a bug. Run it, read the last line of the error, then use Gemini's explain/fix option.
Before you accept the fix, make sure you understand *why* it failed. Then fix it yourself.

In [ ]:
# This cell has a deliberate bug. Run it, read the error, then fix it.
average_temp = sum(bearing_temp_c) / len(bearing_temps)
print(f"Average bearing temperature: {average_temp:.1f} C")

### Exercise 0.10 [Level 3: write it yourself, Gemini allowed]

Write a function `convert(value, from_unit, to_unit)` that handles **W to kW**, **K to degC** and **F to degC**.
If it is asked for a conversion it does not know, it should `raise ValueError` with a helpful message rather than
guessing. (Silent guessing is how unit errors get into reports.)

**Prompt to try in Gemini** (Learn Mode on if you want guidance rather than an answer):
> I need a Python function convert(value, from_unit, to_unit) that supports W to kW, K to degC and F to degC,
> and raises ValueError for anything else. Explain each line. What are you assuming?

Then run the check cell. If it fails, work out why before asking again.

In [ ]:
# TODO: write your convert() function here (you may reuse the functions you wrote above)

In [ ]:
ok = convert(2500, "W", "kW") == 2.5 and abs(convert(373.15, "K", "degC") - 100) < 1e-9 \
     and abs(convert(32, "F", "degC")) < 1e-9
try:
    convert(1, "bar", "psi")
    raised = False
except ValueError:
    raised = True
check(ok and raised, "convert() works and refuses to guess.",
      "Check each conversion, and that an unknown pair raises ValueError.")

## 10. First taste of pandas

`pandas` is the Python library for tables. A table is a **DataFrame**; one column is a **Series**.
We loaded it in the setup cell as `pd`. Here is a small 3D printing test dataset.

In [ ]:
printer = pd.read_csv("data/3dprinter.csv")
printer.head()

### Exercise 0.11 [Level 1: fill the blanks]

How many rows and columns? How many tests used each material? What is the average elongation for each material?

In [ ]:
print("Rows, columns:", printer.____)
print(printer["____"].value_counts())
printer.groupby("material")["____"].____()

`groupby` + `mean` is the pandas version of a pivot table. You will use it a lot.

Try the same thing with `"roughness"` instead of `"elongation"` in a new cell. It fails. Why would a column of
surface roughness numbers refuse to average? Hold that thought: it is the first thing we look at in Module 01.

## Wrap-up: the syntax you need for the rest of the day

| You want to | Python |
|---|---|
| Store a value | `power_kw = 15.0` |
| Format text with a value | `f"{power_kw:.1f} kW"` |
| Tidy text | `"  cnc-009 ".strip().upper()` |
| A list / first item / last item | `temps = [61, 63]`, `temps[0]`, `temps[-1]` |
| A dictionary / read a key | `asset = {"site": "Cork"}`, `asset["site"]` |
| Loop with a decision | `for t in temps:` then indented `if t > 65:` |
| Reusable calculation | `def f(x):` ... `return` |
| Read a CSV | `pd.read_csv("data/file.csv")` |
| Count values / average by group | `df["col"].value_counts()`, `df.groupby("a")["b"].mean()` |

Next: **01 Engineering Data Reality Check**.